In [19]:
import json
import os
import shutil
import pandas as pd
from B1_UTILS import download_map_model, map_normalizing, map_output
from B2_UTILS_data2npy import data_to_npy

In [20]:
DOWNLOAD = False
NORMALIZATION = False
MAP2NPY = False

In [21]:
MODEL_PARTS = ["unit_1", "unit_2"]

MAIN_HOME_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net'
PATH_SETTINGS = {
    "Filtered_Dateset": [
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Raw'),
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Training'),
        os.path.join(MAIN_HOME_PATH, 'Filtered_Dateset', 'Raw', 'Final_Filtered_Data.csv'),
    ],
}

PATH_KEYS = "Filtered_Dateset"  # "training3" # Possible values: "training_data", "test_ribosome_cryoET", "test_ribosome_cryoEM"
DATA_PATH, HOME_PATH, csv_path = PATH_SETTINGS[PATH_KEYS]
temp_sample_path = os.path.join(HOME_PATH, "ready_to_train_and_val")
os.makedirs(temp_sample_path, exist_ok=True)

In [22]:
# Read map list and generate raw_map and model downloading paths
df = pd.read_csv(csv_path)
emdbs, pdbs = df["emdb_id"], df["fitted_pdbs"]
resolutions = df["resolution"].astype(str)
emdb_ids = [emdb.split("-")[1] for emdb in emdbs]
folders = [f"{emdb}_re_{resolution}" for emdb, resolution in zip(emdbs, resolutions)]
raw_maps = [f"emd_{emdb_id}.map" for emdb_id in emdb_ids]
models = [f"{pdb}.cif" for pdb in pdbs]
raw_map_paths = [f"{DATA_PATH}/{folder}/{raw_map}" for folder, raw_map in zip(folders, raw_maps)]
model_paths = [f"{DATA_PATH}/{folder}/{model}" for folder, model in zip(folders, models)]

In [ ]:
map_paths = [f"{raw_map_path.split('.map')[0]}_normalized.mrc" for raw_map_path in raw_map_paths]
for idx, emdb_id in enumerate(emdb_ids):
    if DOWNLOAD:
        download_map_model(emdb=emdbs[idx], pdb=pdbs[idx], resolution=resolutions[idx], directory=DATA_PATH)
        # if idx == 20:   # for download limited number of testing data
        #     exit()
    if NORMALIZATION and not os.path.exists(map_paths[idx]):
        map_data = map_normalizing(raw_map_paths[idx])
        map_output(raw_map_paths[idx], map_data, map_paths[idx], is_model = False)


In [28]:
DOWNLOAD = False
NORMALIZATION = False
MAP2NPY = True

In [29]:
if MAP2NPY:
    # Create training and testing dataset (3d numpy arraies) from map file and model file
    sample_num = 0  # idx of npy file
    num_of_tag_in_each_part_for_all_models = {
    }  # number of tag in each part for all models

    # for idx, emdb_id in enumerate(emdb_ids):
    for idx, emdb_id in enumerate(emdb_ids[0:2]):
        # for idx in [0, 1]:
        print(f"Generating dataset from EMDB-{emdb_id}... ")
        sample_num, num_of_tag_in_each_part = data_to_npy(
            map_paths[idx], model_paths[idx], MODEL_PARTS, temp_sample_path,
            sample_num)




Generating dataset from EMDB-11893... 
Sampling part: unit_1
Sampling part: unit_2
Generating dataset from EMDB-22345... 
Sampling part: unit_1
Sampling part: unit_2


In [30]:
print([
    map_paths[idx], model_paths[idx], MODEL_PARTS, temp_sample_path, sample_num
])

['/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-22345_re_2.8/emd_22345_normalized.mrc', '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw/EMD-22345_re_2.8/7jil.cif', ['unit_1', 'unit_2'], '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/ready_to_train_and_val', 2]
